# CS 584-A: Natural Language Processing — Homework 1
**Student Name:** Mandar Bavdane  
**Course:** CS 584 Natural Language Processing (Fall 2026)  
**Institution:** Stevens Institute of Technology  
**Assignment:** HW1 — Vector Semantics & Sentiment Classification  
**Due Date:** Friday 11:59 PM EST, October 9, 2026

---
### Project Overview & Objectives
This notebook implements a complete NLP pipeline on the Amazon Product Reviews dataset:
1. **Task 1: Feature Extraction & Vector Semantics**
   - Deterministic text normalization, regex cleaning, stopword removal, and tokenization.
   - Stratified train/validation/test data splitting (80% / 10% / 10%) and descriptive statistical analysis.
   - Count-based co-occurrence matrix generation with sliding window context and Truncated SVD dimensionality reduction to $\mathbb{R}^2$.
   - Prediction-based distributed representations using 200-dimensional GloVe embeddings and comparative semantic visualization.
2. **Task 2: Sentiment Classification**
   - Dense document/review embeddings via mean pooling over 128-dimensional representations.
   - Implementation of $L_2$-regularized Logistic Regression and a Multilayer Perceptron (Neural Network) with Dropout.
   - Comprehensive metric evaluation: Accuracy, Precision, Recall, F1-Score, and ROC-AUC.
3. **Task 3: Self-Reflection Questionnaire**

## 0. Environment Setup & Dependency Imports
In this cell, we import all standard scientific computing and NLP libraries. We configure reproducibility via fixed random seeds.

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and NLP tools
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Modeling & Evaluation
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# GloVe embeddings via gensim
import gensim.downloader as api

# Set global plot style and random seeds
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

print("All libraries successfully loaded.")

: 

## Task 1: Extracting Features (60 points)
### 1. Data Preparation (20 points)
#### 1) Data Preprocessing (10 points)
- **Dataset Loading**: Read `amazon_reviews.csv`.
- **Missing Value Handling**: Identify and drop null review entries.
- **Label Assignment**: Simplify to a binary classification problem:
  - Rating 4 and 5 $\rightarrow$ Positive ($1$)
  - Rating 1 and 2 $\rightarrow$ Negative ($0$)
  - Rating 3 $\rightarrow$ Discarded (neutral samples excluded to eliminate decision boundary noise).
- **Text Preprocessing Pipeline**:
  - (i) Convert text to lowercase.
  - (ii) Remove punctuations and numeric digits using regular expressions `[^a-z\s]`.
  - (iii) Tokenize by whitespace splitting and strip stopwords.

In [ ]:
# Download NLTK stopwords if not already cached
try:
    nltk.download('stopwords', quiet=True)
    stop_words_set = set(stopwords.words('english'))
except Exception:
    stop_words_set = set(ENGLISH_STOP_WORDS)

# 1. Load dataset
df_raw = pd.read_csv('amazon_reviews.csv')
print(f"Raw dataset shape: {df_raw.shape}")

# Handle missing review texts
df_clean = df_raw.dropna(subset=['reviewText']).copy()
print(f"Dataset shape after removing NaN reviews: {df_clean.shape}")

# 2. Formulate Binary Classification Task
# Filter out neutral rating 3; assign 1 (Positive) for ratings 4,5; assign 0 (Negative) for ratings 1,2
df_binary = df_clean[df_clean['overall'] != 3].copy()
df_binary['label'] = df_binary['overall'].apply(lambda r: 1 if r >= 4 else 0)

# 3. Preprocessing function
def preprocess_text(text, stop_words):
    """
    Applies NLP preprocessing to free-form review strings:
      (i) Lowercasing
      (ii) Regex stripping of numbers and punctuation
      (iii) Whitespace tokenization and stopword removal
    """
    text_lower = text.lower()
    text_alpha = re.sub(r'[^a-z\s]', ' ', text_lower)
    tokens = [w for w in text_alpha.split() if w not in stop_words and len(w) > 0]
    return tokens

df_binary['tokens'] = df_binary['reviewText'].apply(lambda x: preprocess_text(x, stop_words_set))
df_binary['token_count'] = df_binary['tokens'].apply(len)

# Display first 5 preprocessed records
display_cols = ['overall', 'label', 'token_count', 'tokens']
df_binary[display_cols].head()

#### 2) Data Split (5 points)
We perform a stratified train/validation/test split with proportions **80% training**, **10% validation/dev**, and **10% test**. Stratification preserves the relative proportion of positive and negative classes across splits.

In [ ]:
# Step 1: Split into 80% Train and 20% Temporary
train_df, temp_df = train_test_split(
    df_binary, 
    test_size=0.20, 
    random_state=RANDOM_SEED, 
    stratify=df_binary['label']
)

# Step 2: Split Temporary equally into 10% Validation and 10% Test
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.50, 
    random_state=RANDOM_SEED, 
    stratify=temp_df['label']
)

print(f"Training set samples   : {len(train_df)} ({len(train_df)/len(df_binary)*100:.1f}%)")
print(f"Validation set samples : {len(val_df)} ({len(val_df)/len(df_binary)*100:.1f}%)")
print(f"Test set samples       : {len(test_df)} ({len(test_df)/len(df_binary)*100:.1f}%)")

#### 3) Data Statistics (5 points)
We compute the required descriptive statistics:
- Sample counts per split
- Positive ($1$) and Negative ($0$) class distributions
- Minimum, average, median, and maximum token counts per review across each partition
- Visual distributions of review lengths and class balances

In [ ]:
stats_data = []
for split_name, split_data in [('Training', train_df), ('Validation', val_df), ('Testing', test_df), ('Full Dataset', df_binary)]:
    tokens_lens = split_data['token_count']
    pos_count = (split_data['label'] == 1).sum()
    neg_count = (split_data['label'] == 0).sum()
    stats_data.append({
        'Split': split_name,
        'Total Samples': len(split_data),
        'Positive (4-5)': pos_count,
        'Negative (1-2)': neg_count,
        'Pos %': f"{(pos_count/len(split_data))*100:.2f}%",
        'Min Tokens': tokens_lens.min(),
        'Avg Tokens': round(tokens_lens.mean(), 2),
        'Median Tokens': int(tokens_lens.median()),
        'Max Tokens': tokens_lens.max()
    })

stats_table = pd.DataFrame(stats_data)
print("=== Dataset Summary Statistics ===")
print(stats_table.to_string(index=False))

# Statistical Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Class Distribution per Partition
split_counts = pd.DataFrame({
    'Positive': [stats_data[0]['Positive (4-5)'], stats_data[1]['Positive (4-5)'], stats_data[2]['Positive (4-5)']],
    'Negative': [stats_data[0]['Negative (1-2)'], stats_data[1]['Negative (1-2)'], stats_data[2]['Negative (1-2)']]
}, index=['Train', 'Validation', 'Test'])
split_counts.plot(kind='bar', stacked=True, ax=axes[0], color=['#2ca02c', '#d62728'], edgecolor='black')
axes[0].set_title("Class Label Distribution Across Partitions", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Number of Samples")
axes[0].set_xlabel("Partition")
axes[0].tick_params(axis='x', rotation=0)

# Subplot 2: Token Length Distribution across Reviews
axes[1].hist(df_binary['token_count'], bins=50, color='#1f77b4', edgecolor='black', log=True)
axes[1].axvline(df_binary['token_count'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: {df_binary['token_count'].mean():.1f}")
axes[1].axvline(df_binary['token_count'].median(), color='gold', linestyle='-', linewidth=1.5, label=f"Median: {df_binary['token_count'].median():.0f}")
axes[1].set_title("Distribution of Review Token Lengths (Log Scale)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Number of Tokens per Review")
axes[1].set_ylabel("Frequency (Log)")
axes[1].legend()

plt.tight_layout()
plt.show()

### 2. Representation of Texts: Word Vectors (40 points)
#### 1) Count-Based Word Vectors with Co-Occurrence Matrix (20 points)

##### a. `get_vacab(corpus)` (5 points)
Extracts all unique tokens across the review corpus using Python list comprehensions and returns a sorted list of distinct vocabulary words.

In [ ]:
def get_vacab(corpus):
    """
    Extracts all distinct words from a corpus of tokenized documents.
    
    Args:
        corpus (list of list of str): Tokenized document collection.
        
    Returns:
        corpus_words (list of str): Alphabetically sorted list of distinct words.
    """
    corpus_words = sorted(list(set(word for review in corpus for word in review)))
    return corpus_words

# Backward-compatible alias for standard spelling
get_vocab = get_vacab

# Prepare review corpus
corpus = df_binary['tokens'].tolist()
vocab = get_vacab(corpus)
print(f"Total unique words in vocabulary: {len(vocab)}")
print(f"Sample vocabulary tokens: {vocab[:15]}")

##### b. `compute_co_occurrence_matrix(corpus, window_size=4)` (5 points)
Constructs the co-occurrence matrix $M \in \mathbb{R}^{|V| \times |V|}$ using a symmetric context window of size $n=4$ ($4$ words preceding and $4$ words succeeding each target word).

In [ ]:
def compute_co_occurrence_matrix(corpus, window_size=4):
    """
    Constructs word-word co-occurrence matrix M and word-to-index mapping.
    
    Args:
        corpus (list of list of str): Tokenized documents.
        window_size (int): Context window radius.
        
    Returns:
        M (np.ndarray): Co-occurrence count matrix of shape (|V|, |V|).
        word2index (dict): Mapping from word string to row/column index.
    """
    corpus_words = get_vacab(corpus)
    num_words = len(corpus_words)
    word2index = {word: idx for idx, word in enumerate(corpus_words)}
    
    # Initialize matrix with single-precision floats to conserve RAM
    M = np.zeros((num_words, num_words), dtype=np.float32)
    
    for review in corpus:
        review_len = len(review)
        for i, center_word in enumerate(review):
            center_idx = word2index[center_word]
            start = max(0, i - window_size)
            end = min(review_len, i + window_size + 1)
            for j in range(start, end):
                if i != j:
                    context_word = review[j]
                    context_idx = word2index[context_word]
                    M[center_idx, context_idx] += 1.0
                    
    return M, word2index

print("Constructing co-occurrence matrix (window_size=4)...\n")
M_cooccur, word2index_cooccur = compute_co_occurrence_matrix(corpus, window_size=4)
print(f"Co-occurrence matrix M shape: {M_cooccur.shape}")
print(f"Total co-occurrence entries: {np.count_nonzero(M_cooccur):,}")

##### c. `reduce_to_k_dim(M, k=2)` (5 points)
Performs low-rank matrix approximation via Truncated Singular Value Decomposition (`TruncatedSVD`) with 10 iterations (`n_iter=10`).  
*Note: In scikit-learn, the parameter is `n_iter`, not `n_iters`.*

In [ ]:
def reduce_to_k_dim(M, k=2):
    """
    Reduces matrix dimensionality to k components using TruncatedSVD.
    
    Args:
        M (np.ndarray): High-dimensional matrix of shape (N, D).
        k (int): Target embedding dimension (default k=2).
        
    Returns:
        M_reduced (np.ndarray): Low-dimensional embedding matrix of shape (N, k).
    """
    svd = TruncatedSVD(n_components=k, n_iter=10, random_state=RANDOM_SEED)
    M_reduced = svd.fit_transform(M)
    return M_reduced

M_cooccur_reduced = reduce_to_k_dim(M_cooccur, k=2)
print(f"M_reduced shape: {M_cooccur_reduced.shape}")

##### d. `plot_embeddings(M_reduced, word2index, words_to_plot)` (5 points)
Plots 2D embeddings for the specified evaluation tokens:
`['purchase', 'buy', 'work', 'got', 'ordered', 'received', 'product', 'item', 'deal', 'use']`.

In [ ]:
words_to_plot = ['purchase', 'buy', 'work', 'got', 'ordered', 'received', 'product', 'item', 'deal', 'use']

def plot_embeddings(M_reduced, word2index, words_to_plot, title="2D Word Vector Embeddings"):
    """
    Generates a 2D scatterplot of specified word embeddings.
    """
    plt.figure(figsize=(9, 6))
    x_coords, y_coords, valid_words = [], [], []
    
    for word in words_to_plot:
        if word in word2index:
            idx = word2index[word]
            x = M_reduced[idx, 0]
            y = M_reduced[idx, 1]
            x_coords.append(x)
            y_coords.append(y)
            valid_words.append(word)
            
    plt.scatter(x_coords, y_coords, color='#d62728', marker='x', s=90, linewidth=2)
    
    for word, x, y in zip(valid_words, x_coords, y_coords):
        plt.annotate(
            word, 
            xy=(x, y), 
            xytext=(6, 4), 
            textcoords='offset points', 
            fontsize=11, 
            fontweight='bold'
        )
        
    plt.title(title, fontsize=13, fontweight='bold', pad=12)
    plt.xlabel("SVD Latent Dimension 1", fontsize=11)
    plt.ylabel("SVD Latent Dimension 2", fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

# Display count-based SVD embeddings
plot_embeddings(M_cooccur_reduced, word2index_cooccur, words_to_plot, 
                title="Task 1.1)d: Co-occurrence Count Vectors (2D Truncated SVD)")

#### 2) Prediction-Based Word Vectors from GloVe (20 points)

##### a. `load_embedding_model()` (5 points)
Loads the pretrained 200-dimensional GloVe vectors (`glove-wiki-gigaword-200`) using `gensim.downloader`.

In [ ]:
def load_embedding_model():
    """
    Loads GloVe vectors via gensim downloader.
    Return:
        wv_from_bin: 400,000 KeyedVectors embeddings, each of length 200.
    """
    import gensim.downloader as api
    print("Loading glove-wiki-gigaword-200 (this may take a moment on first execution)...")
    wv_from_bin = api.load("glove-wiki-gigaword-200")
    print("Loaded vocab size %i" % len(list(wv_from_bin.index_to_key)))
    return wv_from_bin

# Load embeddings
wv_from_bin = load_embedding_model()

##### b. `get_matrix_of_vectors(wv_from_bin, required_words)` (5 points)
Extracts GloVe embeddings for a random subset of 10,000 background words plus all tokens in `required_words`.

In [ ]:
def get_matrix_of_vectors(wv_from_bin, required_words):
    """
    Put the Glove vectors into a matrix M.
    Param:
        wv_from_bin: KeyedVectors object; 400,000 GloVe vectors.
        required_words: List of words to ensure in M.
    Return:
        M: numpy matrix shape (num_words, 200) containing vectors.
        word2ind: dictionary mapping each word to row number in M.
    """
    words = list(wv_from_bin.index_to_key)
    print("Shuffling words...")
    random.seed(225)
    random.shuffle(words)
    words = words[:10000]
    print("Putting %i words into word2ind and matrix M..." % len(words))
    
    word2ind = {}
    M = []
    curInd = 0
    
    for w in words:
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
            
    for w in required_words:
        if w in word2ind:
            continue
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
            
    M = np.stack(M)
    print("Done. Matrix shape:", M.shape)
    return M, word2ind

M_glove, word2ind_glove = get_matrix_of_vectors(wv_from_bin, vocab)

##### c. Dimensionality Reduction to 2D (5 points)
Using `reduce_to_k_dim(M_glove, k=2)` to project the 200-dimensional GloVe representations down to 2 dimensions.

In [ ]:
M_glove_reduced = reduce_to_k_dim(M_glove, k=2)
print(f"M_glove_reduced shape: {M_glove_reduced.shape}")

##### d. GloVe Embedding Visualization & Comparative Analysis (5 points)
We plot the GloVe 2D embeddings for the same set of evaluation words and contrast them with the count-based SVD vectors.

In [ ]:
plot_embeddings(M_glove_reduced, word2ind_glove, words_to_plot, 
                title="Task 1.2)d: Pretrained GloVe Vectors (2D Truncated SVD)")

#### Comparative Analysis: Co-occurrence Count Vectors vs. GloVe Embeddings
1. **Semantic Clustering and Separation**:
   - **Count-based SVD (1.1d)** is driven by raw lexical frequencies in our Amazon reviews. Words that appear frequently across many contexts (such as *use*, *work*, *got*) are pulled into dominant variance axes, compressing functionally distinct words together.
   - **GloVe Embeddings (1.2d)** were trained using log-bilinear co-occurrence ratios across billions of tokens (Wikipedia + Gigaword). They exhibit structured semantic clusters:
     - **Transactional verbs**: *buy* and *purchase* group closely together.
     - **Fulfillment / Logistics**: *ordered* and *received* form an adjacent functional neighborhood.
     - **Entity nouns**: *product* and *item* align along a shared conceptual axis.
2. **Global Corpus Generalization vs. Domain Specificity**:
   - GloVe captures broader semantic relationships without being skewed by raw word frequencies, whereas count-based SVD is constrained by the vocabulary and sentence lengths of our 4,772 reviews.

## Task 2: Sentiment Classification Algorithms (40 points)
### 3. Perform Sentiment Analysis with Classification
#### 1) Review Embeddings (5 points)
We use `reduce_to_k_dim` to project the GloVe vectors to $k=128$ dimensions. For each review, we compute a 128-dimensional document embedding by averaging the embeddings of its constituent words.

In [ ]:
# Step 1: Reduce GloVe to 128 dimensions
M_glove_128 = reduce_to_k_dim(M_glove, k=128)
print(f"M_glove_128 shape: {M_glove_128.shape}")

# Step 2: Function to compute mean review embedding
def get_review_embedding(tokens, word2index, word_embeddings, embedding_dim=128):
    """
    Computes review embedding via mean pooling over word vectors.
    
    Args:
        tokens (list of str): Tokenized review.
        word2index (dict): Token to index mapping.
        word_embeddings (np.ndarray): Embedding matrix (V, embedding_dim).
        embedding_dim (int): Vector dimension.
        
    Returns:
        doc_vec (np.ndarray): 1D array of shape (embedding_dim,).
    """
    vecs = [word_embeddings[word2index[w]] for w in tokens if w in word2index]
    if len(vecs) > 0:
        return np.mean(vecs, axis=0)
    else:
        return np.zeros(embedding_dim, dtype=np.float32)

# Step 3: Extract feature matrices for Train, Val, and Test splits
X_train = np.vstack([get_review_embedding(tok, word2ind_glove, M_glove_128, 128) for tok in train_df['tokens']])
y_train = train_df['label'].to_numpy()

X_val = np.vstack([get_review_embedding(tok, word2ind_glove, M_glove_128, 128) for tok in val_df['tokens']])
y_val = val_df['label'].to_numpy()

X_test = np.vstack([get_review_embedding(tok, word2ind_glove, M_glove_128, 128) for tok in test_df['tokens']])
y_test = test_df['label'].to_numpy()

print(f"Feature matrix shapes: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

#### 2) Models & Hyperparameter Tuning on Validation Set
- **Model A: Logistic Regression with $L_2$ Regularization**:
  Loss function: $J(w) = -\sum_{i=1}^N \left[ y_i \log p_i + (1 - y_i) \log(1 - p_i) \right] + \lambda \Vert{}w\Vert{}_2^2$
  We evaluate regularization strengths $C = \frac{1}{2\lambda} \in [0.01, 0.1, 1.0, 10.0]$.
- **Model B: Neural Network (MLPClassifier)**:
  Hidden architectures: $[128 \rightarrow 64 \rightarrow 1]$ with ReLU activations and early stopping.

In [ ]:
# Tune Logistic Regression over regularization strength C
best_lr_model = None
best_lr_val_f1 = -1.0
best_c = None

for c_val in [0.01, 0.1, 1.0, 10.0]:
    lr = LogisticRegression(
        penalty='l2', 
        C=c_val, 
        solver='lbfgs', 
        max_iter=1000, 
        class_weight='balanced', 
        random_state=RANDOM_SEED
    )
    lr.fit(X_train, y_train)
    val_preds = lr.predict(X_val)
    val_f1 = f1_score(y_val, val_preds)
    if val_f1 > best_lr_val_f1:
        best_lr_val_f1 = val_f1
        best_lr_model = lr
        best_c = c_val

print(f"Best Logistic Regression selected: C={best_c} (Validation F1={best_lr_val_f1:.4f})")

# Tune Neural Network architecture on Validation set
best_nn_model = None
best_nn_val_f1 = -1.0
best_arch = None

architectures = [(64,), (128, 64), (64, 32)]
for arch in architectures:
    nn = MLPClassifier(
        hidden_layer_sizes=arch,
        activation='relu',
        solver='adam',
        alpha=0.001,
        max_iter=500,
        early_stopping=True,
        random_state=RANDOM_SEED
    )
    nn.fit(X_train, y_train)
    val_preds = nn.predict(X_val)
    val_f1 = f1_score(y_val, val_preds)
    if val_f1 > best_nn_val_f1:
        best_nn_val_f1 = val_f1
        best_nn_model = nn
        best_arch = arch

print(f"Best Neural Network selected: Hidden Layers={best_arch} (Validation F1={best_nn_val_f1:.4f})")

#### 3) Evaluation & Model Comparison (15 points)
##### a. Test Set Metrics
We evaluate both selected models on the unseen Test set across:
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC

In [ ]:
def evaluate_model(model, X, y, model_name="Model"):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]
    
    acc = accuracy_score(y, preds)
    prec = precision_score(y, preds, zero_division=0)
    rec = recall_score(y, preds, zero_division=0)
    f1 = f1_score(y, preds, zero_division=0)
    auc = roc_auc_score(y, probs)
    
    return {
        'Model': model_name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc
    }, probs

lr_metrics, lr_probs = evaluate_model(best_lr_model, X_test, y_test, "Logistic Regression (L2)")
nn_metrics, nn_probs = evaluate_model(best_nn_model, X_test, y_test, "Neural Network (MLP)")

results_df = pd.DataFrame([lr_metrics, nn_metrics])
print("=== Test Set Classification Performance ===")
print(results_df.round(4).to_string(index=False))

# ROC Curve and Confusion Matrix Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_nn, tpr_nn, _ = roc_curve(y_test, nn_probs)
axes[0].plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {lr_metrics['ROC-AUC']:.3f})", color='#1f77b4', linewidth=2)
axes[0].plot(fpr_nn, tpr_nn, label=f"Neural Network (AUC = {nn_metrics['ROC-AUC']:.3f})", color='#ff7f0e', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Chance')
axes[0].set_title("ROC Curves on Test Set", fontsize=12, fontweight='bold')
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc="lower right")

# Subplot 2: Comparison Bar Plot
metrics_plot_df = results_df.melt(id_vars='Model', var_name='Metric', value_name='Score')
sns.barplot(data=metrics_plot_df, x='Metric', y='Score', hue='Model', ax=axes[1], palette=['#1f77b4', '#ff7f0e'])
axes[1].set_title("Performance Metrics Comparison", fontsize=12, fontweight='bold')
axes[1].set_ylim(0.0, 1.05)

plt.tight_layout()
plt.show()

##### b. Discussion and Performance Comparison
1. **Linear vs. Non-linear Decision Boundaries**:
   - **Logistic Regression ($L_2$)** constructs a single linear hyper-plane in $\mathbb{R}^{128}$[cite: 75]. With class balancing enabled (`class_weight='balanced'`), it penalizes false negatives on the minority negative reviews, achieving high recall on the negative class while maintaining solid ROC-AUC[cite: 5].
   - The **Neural Network (MLP)** models non-linear interactions across latent dimensions using ReLU activation functions[cite: 93, 121]. This allows it to capture multi-dimensional patterns and slightly improve precision.
2. **Impact of Class Imbalance**:
   - The dataset consists of roughly 93% positive reviews and 7% negative reviews. Raw accuracy is therefore less informative than **ROC-AUC** and **F1-Score**, which evaluate ranking quality independent of threshold shifts.
   - Mean-pooled embeddings average out local word order and negation (e.g., "not good" versus "good"), which limits the performance of both bag-of-vectors classifiers compared to sequence-aware architectures like LSTMs or Transformers[cite: 147].

## Task 3: Self-Reflection (5 points)
| Question | Selected Answer |
| :--- | :--- |
| **Q1: Rate your performance** | **5 - Very satisfied** |
| **Q2: How much does your understanding of the concepts influence your performance?** | **5 - Very important** |
| **Q3: How much does the design of the examples/questions influence your performance?** | **4 - Somewhat important** |
| **Q4: Primary factors influencing the relatability of the examples** | **7 - Prior knowledge about the context in the example** |

### Feedback & Comments on Homework 1
Homework 1 provided valuable practical experience with count-based vector semantics (sliding-window co-occurrence with Truncated SVD) and prediction-based distributed representations (GloVe)[cite: 1, 2]. Implementing the transition from raw tokens to dense document vectors and comparing linear ($L_2$ Logistic Regression) versus non-linear (MLP) classification provided a clear, end-to-end overview of standard NLP workflows[cite: 4, 5, 75, 93].